# Train LLaVA Pruner on Colab

This notebook trains only the pruner using the saved provenance samples in `dataset/samples/`.
It assumes the repo is available on Google Drive and runs well on a single A100.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U torch torchvision transformers accelerate "bitsandbytes>=0.46.1" "datasets>=3.6.0" sentencepiece pillow tqdm

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

CONFIG = {
    # CHANGE THESE TO YOUR ENVIRONMENT
    "repo_dir": "/content/drive/MyDrive/dynamic_pruning/DynamicVisualTokenPruning",
    "samples_dir": "dataset/samples",
    "output_dir": "outputs/colab_pruner_training",
    "feature_cache_dir": "dataset/feature_cache",

    # TRAINING PARAMS
    "model_id": "llava-hf/llava-1.5-7b-hf",
    "keep_ratio": 0.5,
    "target_layer": 7,
    "train_split": 0.9,
    "epochs": 5,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "soft_target_weight": 0.25,
    "seed": 42,

    # PRUNER PARAMS
    "num_heads": 8,
    "single_head": False,
    "load_in_4bit": True,
    "trust_remote_code": True,
    "max_train_samples": 0,
    "max_val_samples": 0,
}

repo_dir = Path(CONFIG["repo_dir"]).expanduser().resolve()
assert repo_dir.exists(), f"Missing repo_dir: {repo_dir}"
assert (repo_dir / "train_pruner.py").exists(), "train_pruner.py not found"

print("Python:", sys.executable)
print("Repo dir:", repo_dir)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
subprocess.check_call([
    sys.executable,
    "-c",
    "import torch; print('cuda', torch.cuda.is_available()); print('device_count', torch.cuda.device_count()); print('device_name', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')",
], cwd=repo_dir)

In [ ]:
def resolve_path(repo_dir, maybe_relative):
    path = Path(maybe_relative)
    return path if path.is_absolute() else repo_dir / path

train_script = repo_dir / "train_pruner.py"
samples_dir = resolve_path(repo_dir, CONFIG["samples_dir"])
output_dir = resolve_path(repo_dir, CONFIG["output_dir"])
feature_cache_dir = resolve_path(repo_dir, CONFIG["feature_cache_dir"])

output_dir.mkdir(parents=True, exist_ok=True)
feature_cache_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(train_script),
    "--samples-dir", str(samples_dir),
    "--output-dir", str(output_dir),
    "--feature-cache-dir", str(feature_cache_dir),
    "--model-id", CONFIG["model_id"],
    "--keep-ratio", str(CONFIG["keep_ratio"]),
    "--target-layer", str(CONFIG["target_layer"]),
    "--train-split", str(CONFIG["train_split"]),
    "--epochs", str(CONFIG["epochs"]),
    "--lr", str(CONFIG["lr"]),
    "--weight-decay", str(CONFIG["weight_decay"]),
    "--soft-target-weight", str(CONFIG["soft_target_weight"]),
    "--seed", str(CONFIG["seed"]),
    "--num-heads", str(CONFIG["num_heads"]),
]

if CONFIG["single_head"]:
    cmd.append("--single-head")
if CONFIG["load_in_4bit"]:
    cmd.append("--load-in-4bit")
if CONFIG["trust_remote_code"]:
    cmd.append("--trust-remote-code")
if CONFIG["max_train_samples"] > 0:
    cmd.extend(["--max-train-samples", str(CONFIG["max_train_samples"])])
if CONFIG["max_val_samples"] > 0:
    cmd.extend(["--max-val-samples", str(CONFIG["max_val_samples"])])

print("Launching:\n" + " ".join(shlex.quote(part) for part in cmd))

In [ ]:
subprocess.check_call(cmd, cwd=repo_dir)

In [ ]:
print("Outputs written to:", output_dir)
print("Checkpoint:", output_dir / "best_pruner.pt")
print("Metrics:", output_dir / "metrics.json")